# Huấn luyện SAC mới trên GPU

Đăng nhập tài khoản Google mới của Trường. Chọn **Thời gian chạy →
Thay đổi loại thời gian chạy → GPU (T4 nếu có)** trước khi chạy các ô.
Khi kết nối Google Drive, chọn đúng tài khoản mới.

Bản này dùng thư mục SAC_GPU_20261127_v1, STAGE='all' và buộc CUDA.
Ô 3 kiểm tra GPU thật; nếu chưa có GPU hoặc PyTorch chỉ hỗ trợ CPU,
notebook dừng với hướng dẫn rõ ràng, không chuyển sang huấn luyện CPU.

Quy trình: pilot mới 50.000 bước × 2 seed × 2 cấu hình → chọn cấu hình
bằng validation → phân tích S2/S3 → huấn luyện mới 300.000 bước × 5 seed
→ tổng hợp validation → đánh giá test. Không nạp trọng số từ lần chạy cũ.

Log trực tiếp và đồng hồ xuất hiện trong lúc huấn luyện. Kết quả được lưu
trên Drive; nếu Colab ngắt phiên, chạy lại từ đầu với cùng tên thí nghiệm
và runtime GPU để tiếp tục các snapshot của chính lần chạy mới này.

[Giới hạn tài nguyên Colab](https://research.google.com/colaboratory/faq.html).
GPU do Colab cấp; notebook không tự cấp phát được GPU.


In [ ]:
# 1. Kết nối Drive của tài khoản mới và tạo thư mục thí nghiệm GPU riêng.
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

# Giữ nguyên tên này khi tiếp tục sau khi ngắt phiên.
# Muốn một lần chạy độc lập khác, đổi hậu tố v1 thành v2 trước khi chạy.
EXPERIMENT_NAME = 'SAC_GPU_20261127_v1'
STAGE = 'all'  # Chạy pilot, final và test theo đúng thứ tự.
DEVICE_REQUEST = 'cuda'  # Bản này chỉ huấn luyện trên GPU NVIDIA.
DRIVE_ROOT = Path('/content/drive/MyDrive/SEMS_RL')
OUTPUT = DRIVE_ROOT / EXPERIMENT_NAME
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Thư mục thí nghiệm GPU:', OUTPUT)


In [ ]:
#@title 2. Nạp mã nguồn và CSV đã đóng gói
# Bản giao một file đã có dữ liệu bên trong; không cần GitHub token.
# Notebook mẫu trong repo sẽ yêu cầu tải source_bundle.zip nếu chưa có gói.
import base64
import hashlib
import io
import os
import zipfile

SOURCE_BUNDLE_B64 = ''
EXPECTED_BUNDLE_SHA256 = ''
saved_bundle = OUTPUT / 'source_bundle.zip'
if SOURCE_BUNDLE_B64:
    bundle = base64.b64decode(SOURCE_BUNDLE_B64)
elif saved_bundle.is_file():
    bundle = saved_bundle.read_bytes()
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Chọn đúng một file source_bundle.zip đi kèm notebook.')
    bundle = next(iter(uploaded.values()))
bundle_hash = hashlib.sha256(bundle).hexdigest()
if EXPECTED_BUNDLE_SHA256 and bundle_hash != EXPECTED_BUNDLE_SHA256:
    raise ValueError('Nội dung gói không khớp bản notebook đã giao.')
if saved_bundle.exists() and hashlib.sha256(saved_bundle.read_bytes()).hexdigest() != bundle_hash:
    raise ValueError('Thư mục thí nghiệm đã dùng nguồn khác. Hãy đổi EXPERIMENT_NAME.')
if not saved_bundle.exists():
    temporary = saved_bundle.with_suffix('.tmp')
    temporary.write_bytes(bundle)
    temporary.replace(saved_bundle)
# Chạy mã và đọc CSV trên đĩa runtime, tránh đọc nhiều file nhỏ từ Drive.
WORKSPACE = Path('/content') / ('sems_rl_sac_' + bundle_hash[:12])
WORKSPACE.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(bundle)) as archive:
    for name in archive.namelist():
        target = (WORKSPACE / name).resolve()
        if not target.is_relative_to(WORKSPACE.resolve()):
            raise ValueError('Gói chứa đường dẫn ra ngoài workspace.')
    archive.extractall(WORKSPACE)
os.chdir(WORKSPACE)
print('Đã nạp đúng gói nguồn:', bundle_hash)


In [ ]:
# 3. Cài thư viện và xác nhận PyTorch thực sự thực hiện được phép tính GPU.
# Giữ PyTorch có sẵn của runtime GPU Colab; không cài đè bản CPU.
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', 'requirements-sac.txt'], check=True)
if str(WORKSPACE) not in sys.path:
    sys.path.insert(0, str(WORKSPACE))
import json
import pandas as pd
import torch
from IPython.display import display, Image
import colab_sac as workflow

if STAGE not in ('all', 'pilot', 'final', 'test'):
    raise ValueError('STAGE cần là all, pilot, final hoặc test.')
if DEVICE_REQUEST != 'cuda':
    raise ValueError("Bản chạy GPU này yêu cầu DEVICE_REQUEST='cuda'.")
if torch.version.cuda is None:
    raise RuntimeError(
        'PyTorch hiện tại chỉ hỗ trợ CPU. Hãy chọn runtime GPU, '
        'ngắt kết nối và xóa runtime cũ nếu cần, rồi chạy lại từ đầu '
        'trong runtime GPU sạch; không cài lại bản torch +cpu.')
if not torch.cuda.is_available():
    raise RuntimeError(
        'Chưa có GPU CUDA khả dụng. Vào Thời gian chạy → Thay đổi loại '
        'thời gian chạy → chọn GPU, kết nối lại rồi chạy từ ô 1. '
        'Nếu Colab chưa cấp GPU, cần chờ có GPU; bản này không chạy CPU.')

DEVICE = 'cuda'
gpu_name = torch.cuda.get_device_name(0)
# Phép tính nhỏ xác nhận CUDA dùng được trước khi tạo kế hoạch dài.
probe = torch.ones((8, 8), device=DEVICE)
probe = probe @ probe
torch.cuda.synchronize()
if probe.device.type != 'cuda' or probe[0, 0].item() != 8.0:
    raise RuntimeError('Phép thử CUDA không đạt; chưa bắt đầu huấn luyện.')
del probe

device_file = OUTPUT / 'device_choice.json'
if device_file.is_file():
    saved_device = json.loads(device_file.read_text(encoding='utf-8'))['device']
    if saved_device != DEVICE:
        raise ValueError(
            'Tên thí nghiệm này đã dùng CPU. Chọn EXPERIMENT_NAME mới '
            'để chạy GPU từ đầu; không ghi đè checkpoint CPU.')
plan = workflow.prepare_experiment(OUTPUT)
if not device_file.is_file():
    device_file.write_text(json.dumps({
        'device': DEVICE, 'gpu_name': gpu_name,
        'torch_version': torch.__version__, 'cuda_version': torch.version.cuda,
    }, ensure_ascii=False, indent=2), encoding='utf-8')
print('Thiết bị sử dụng:', DEVICE, '| GPU:', gpu_name,
      '| PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda)
print('Phép thử tính toán GPU: đạt. Các lệnh train_sac.py sẽ dùng --device cuda.')
print('Pilot: 50.000 bước × 2 seed × 2 cấu hình, khởi tạo mới.')
print('Kế hoạch chính thức:', plan['final_steps'], 'bước ×', len(plan['final_seeds']), 'seed')


In [ ]:
"""Đưa stdout/stderr của tiến trình con vào ô Colab và hiển thị thời gian chờ.

Dán nội dung file vào một ô SAU khi import workflow, TRƯỚC ô 8. Chỉ thay hàm
gọi tiến trình trong bộ nhớ; không sửa các file nằm trong khóa mã nguồn SAC.
"""

import csv
import json
import os
import queue
import signal
import subprocess
import sys
import threading
import time
from collections import deque
from functools import partial
from pathlib import Path


def _option(arguments, name):
    values = list(map(str, arguments))
    return values[values.index(name) + 1] if name in values else None


def active_experiment_jobs(workspace, output_root, proc_root=Path('/proc')):
    """Chỉ nhận diện job đúng thư mục/đầu ra này; không dừng tiến trình khác."""
    jobs = []
    if not proc_root.is_dir():
        return jobs  # /proc có trên runtime Linux của Colab.
    for entry in proc_root.iterdir():
        if not entry.name.isdigit() or int(entry.name) == os.getpid():
            continue
        try:
            args = [part.decode('utf-8', errors='replace')
                    for part in (entry / 'cmdline').read_bytes().split(b'\0') if part]
            scripts = {'train_sac.py', 'evaluate_sac.py', 'diagnose_sac.py'}
            if not any(Path(value).name in scripts for value in args):
                continue
            output = _option(args, '--output')
            if output is None or (entry / 'cwd').resolve() != Path(workspace).resolve():
                continue
            target = Path(output)
            if not target.is_absolute():
                target = Path(workspace) / target
            if target.resolve().is_relative_to(Path(output_root).resolve()):
                jobs.append({'pid': int(entry.name), 'output': str(target)})
        except (OSError, ValueError):
            continue  # Tiến trình có thể vừa kết thúc hoặc thuộc người dùng khác.
    return jobs


def training_progress(arguments):
    """Đọc mốc đã GHI RA FILE; không xem đồng hồ sống là bằng chứng mô hình tiến bộ."""
    output = _option(arguments, '--output')
    seed = _option(arguments, '--seeds')
    if not output or seed is None or Path(str(arguments[0])).name != 'train_sac.py':
        return 'đang chờ log từ tiến trình'
    backend = _option(arguments, '--backend') or 'custom'
    run = Path(output) / f'{backend}_seed{seed}'
    parts = [f'seed={seed}']
    for name, label in (('training.csv', 'train'), ('validation.csv', 'val')):
        try:
            with (run / name).open(encoding='utf-8') as stream:
                rows = list(csv.DictReader(stream))
            if rows:
                parts.append(f'{label}={int(rows[-1]["step"]):,} bước')
        except (OSError, KeyError, TypeError, ValueError):
            pass
    try:
        settings = json.loads((run / 'metadata.json').read_text(encoding='utf-8'))['settings']
        parts.append(f'mục tiêu={settings["training"]["total_timesteps"]:,}')
    except (OSError, ValueError, KeyError):
        pass
    if len(parts) == 1:
        parts.append('chưa có CSV tiến độ; đang khởi tạo hoặc đánh giá đầu')
    return ' | '.join(parts)


def _stop_owned_process(process):
    """Dọn đúng nhóm do hàm này tạo khi người dùng bấm Dừng; không pkill hàng loạt."""
    if process.poll() is not None:
        return
    try:
        if os.name == 'posix':
            os.killpg(process.pid, signal.SIGTERM)
        else:
            process.terminate()
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        if os.name == 'posix':
            os.killpg(process.pid, signal.SIGKILL)
        else:
            process.kill()
        process.wait(timeout=5)
    except ProcessLookupError:
        process.wait(timeout=5)


def command_with_logs(*arguments, workspace, output_root, heartbeat_seconds=30):
    """Chuyển log ngay vào notebook; in thời gian và CSV mỗi 30 giây khi chờ.

    Luồng đọc pipe ngăn bộ đệm stdout đầy; luồng đọc CSV tách riêng để việc đọc
    Google Drive chậm không làm đứng đồng hồ trong luồng chính.
    """
    jobs = active_experiment_jobs(workspace, output_root)
    if jobs:
        raise RuntimeError(f'Đã có job của thí nghiệm đang chạy: {jobs}. '
                           'Chưa tạo job mới để tránh hai tiến trình ghi cùng checkpoint.')
    environment = os.environ.copy()
    environment.update({'PYTHONIOENCODING': 'utf-8', 'PYTHONUNBUFFERED': '1'})
    args = [sys.executable, '-u', *map(str, arguments)]
    process = subprocess.Popen(args, cwd=workspace, env=environment,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding='utf-8', errors='replace', bufsize=1,
                               start_new_session=(os.name == 'posix'))
    started, last_heartbeat = time.monotonic(), time.monotonic()
    lines, stopped = queue.Queue(), threading.Event()
    progress = {'text': 'đang khởi tạo; chưa đọc được CSV'}
    tail = deque(maxlen=60)

    def read_stdout():
        try:
            for line in process.stdout:
                lines.put(line)
        finally:
            lines.put(None)

    def read_progress():
        while not stopped.is_set():
            progress['text'] = training_progress(arguments)
            stopped.wait(10)

    reader = threading.Thread(target=read_stdout, daemon=True)
    watcher = threading.Thread(target=read_progress, daemon=True)
    print(f'▶ Bắt đầu {arguments[0]} | PID={process.pid}. '
          f'Thời gian/tiến độ được in mỗi {heartbeat_seconds:g} giây.', flush=True)
    reader.start()
    watcher.start()
    pipe_closed = False
    try:
        while not pipe_closed or process.poll() is None:
            try:
                line = lines.get(timeout=min(1, heartbeat_seconds))
                if line is None:
                    pipe_closed = True
                else:
                    tail.append(line)
                    print(line, end='', flush=True)
            except queue.Empty:
                pass
            now = time.monotonic()
            if now - last_heartbeat >= heartbeat_seconds:
                seconds = int(now - started)
                elapsed = f'{seconds // 3600:02d}:{seconds // 60 % 60:02d}:{seconds % 60:02d}'
                print(f'⏱ {elapsed} | {progress["text"]} | PID={process.pid}', flush=True)
                last_heartbeat = now
        code = process.wait()
        if code:
            raise subprocess.CalledProcessError(code, args, output=''.join(tail))
        print(f'✓ Hoàn tất {arguments[0]} sau {time.monotonic() - started:.1f} giây.', flush=True)
    except KeyboardInterrupt:
        print('Đã nhận lệnh Dừng. Đang dừng đúng tiến trình vừa tạo...', flush=True)
        _stop_owned_process(process)
        print('Chạy lại cùng OUTPUT/DEVICE để tiếp tục từ resume.pt gần nhất.', flush=True)
        raise
    finally:
        stopped.set()
        if process.poll() is None:
            _stop_owned_process(process)
        reader.join(timeout=1)
        process.stdout.close()


def install_colab_logging(workflow, output):
    workflow.command = partial(command_with_logs, workspace=workflow.ROOT, output_root=output)
    print('Đã bật log trực tiếp và bộ đếm thời gian; cấu hình/SAC/checkpoint giữ nguyên.', flush=True)


# Khi dán nguyên file vào Colab, hai biến này đã có từ ô thiết lập notebook.
if 'workflow' in globals() and 'OUTPUT' in globals():
    install_colab_logging(workflow, OUTPUT)


## Dữ liệu và kiểm tra trước khi huấn luyện

Một ngày có 96 bước, mỗi bước 15 phút. Train dùng để học; validation chọn
cấu hình và checkpoint; test chỉ mở sau khi hoàn tất kế hoạch. Không lấy
thống kê validation/test để chuẩn hóa dữ liệu huấn luyện.


In [ ]:
# 4. Xem phân vùng và vài dòng dữ liệu.
from sems_rl.data.data_pipeline import DataPipeline
pipeline = DataPipeline(str(WORKSPACE / 'data/processed.csv'))
split_rows = []
for split, (first, stop) in pipeline.splits.items():
    frame = pipeline.df.iloc[first * 96:stop * 96]
    split_rows.append({'split': split, 'days': stop - first, 'rows': len(frame),
                       'from': frame.iloc[0].timestamp, 'to': frame.iloc[-1].timestamp})
display(pd.DataFrame(split_rows))
display(pipeline.df_raw.head(8))
print('Số ô dữ liệu khuyết:', int(pipeline.df_raw.isna().sum().sum()))


In [ ]:
# 5. Chạy thử nhỏ trước tác vụ dài; chỉ đánh giá validation.
import tempfile
import yaml
from sems_rl.experiments.sac_experiment import read_settings
with tempfile.TemporaryDirectory(prefix='sac_smoke_') as temporary:
    temporary = Path(temporary)
    smoke = read_settings()
    smoke.pop('environment')
    smoke['objective'] = {'mode': 'daily_inventory'}
    smoke['agent'].update({'gamma': 1.0, 'hidden_size': 16, 'batch_size': 8,
                           'buffer_size': 256, 'learning_starts': 32})
    smoke['training'].update({'total_timesteps': 200, 'seeds': [0],
                              'eval_freq': 100, 'eval_days': 1, 'log_freq': 50})
    config = temporary / 'smoke.yaml'
    config.write_text(yaml.safe_dump(smoke), encoding='utf-8')
    workflow.command('train_sac.py', '--config', config, '--device', DEVICE,
                     '--output', temporary / 'models', '--checkpoint-freq', 100)
    workflow.command('evaluate_sac.py', '--runs', temporary / 'models/custom_seed0',
                     '--split', 'val', '--scenarios', 'S2', 'S3', 'S5',
                     '--max-days', 1, '--output', temporary / 'evaluation')
print('Chạy thử hoàn tất. Kết quả thử nhỏ không dùng trong báo cáo chính thức.')


## Phân tích S2/S3 từ pilot mới

Bản GPU không mang theo checkpoint 50.000 bước của lần chạy CPU cũ.
Ô 7 tự huấn luyện các pilot, chọn cấu hình bằng S5 validation rồi dùng
seed 0 của cấu hình được chọn để phân tích hành vi ở S2, S3 và S5.
Huấn luyện chính thức vẫn khởi tạo lại từ đầu ở từng seed.


In [ ]:
# 6. Phân tích hành vi cần checkpoint của pilot vừa huấn luyện.
# Bảng và hình S2/S3 sẽ xuất hiện sau khi ô 7 hoàn tất các pilot mới.
print('Tiếp tục ô 7 để huấn luyện pilot mới trên GPU và phân tích S2/S3.')


## So sánh hai cách đặt mục tiêu rồi khóa cấu hình

- **original:** reward gốc, gamma=0,99, quy ước tiếp nối ngày như bản pilot.
- **daily_inventory:** reward trừ thêm giá trị lượng điện pin dùng ròng ở
  mỗi bước; gamma=1 và kết thúc thật sau 96 bước. Tổng reward một ngày
  đúng bằng âm tổng chi phí đã điều chỉnh chia C_ref.

Mỗi cấu hình chạy 50.000 bước với **seed 0 và 1**. Chọn cấu hình có trung
bình `adjusted_total_cost_vnd` thấp nhất trên **S5 validation**; S2/S3 được
báo cáo để chẩn đoán, không thay tiêu chí sau khi thấy kết quả. Nếu bằng
điểm, ưu tiên original. Đây là so sánh hai cách đặt bài toán, không tách
riêng tác động của gamma, reward và điều kiện kết thúc.

Cấu hình mới là giả thuyết cần kiểm chứng, không mặc định sẽ tốt hơn.


In [ ]:
# 7. Huấn luyện pilot mới, chọn cấu hình rồi phân tích S2/S3.
if STAGE in ('all', 'pilot'):
    print('Pilot GPU: 2 cấu hình × 2 seed, mỗi lần 50.000 bước.', flush=True)
    workflow.run_pilots(OUTPUT, DEVICE)
    comparison, configuration_lock = workflow.lock_configuration(OUTPUT)
    display(comparison)
    print('Cấu hình được chọn:', configuration_lock['selected'])
else:
    if not (OUTPUT / 'configuration_lock.json').is_file():
        raise FileNotFoundError(
            "Chưa có cấu hình đã khóa. Với lần chạy mới, đặt STAGE='all' "
            'ở ô 1, chạy lại ô 1 và ô 7 để huấn luyện pilot trước.')
    configuration_lock = workflow.verify_lock(OUTPUT)
    print('Dùng cấu hình đã khóa:', configuration_lock['selected'])

# Chỉ chẩn đoán trên validation, từ checkpoint thuộc thí nghiệm mới này.
diagnosis = OUTPUT / 'diagnosis_pilot'
if not (diagnosis / 'behavior_summary.csv').is_file():
    if diagnosis.exists() and any(diagnosis.iterdir()):
        import time
        diagnosis.rename(diagnosis.with_name('diagnosis_incomplete_' + str(time.time_ns())))
    pilot_run = OUTPUT / 'pilots' / configuration_lock['selected'] / 'custom_seed0'
    workflow.command('diagnose_sac.py', '--run', pilot_run, '--output', diagnosis)
behavior = pd.read_csv(diagnosis / 'behavior_summary.csv')
display(behavior[['scenario', 'method', 'charge_kwh', 'discharge_kwh',
                  'clipped_at_soc_min', 'wear_cost_vnd', 'final_soc',
                  'adjusted_total_cost_vnd', 'reward_selection_gap_vnd']])
display(Image(filename=str(diagnosis / 'S2_worst_day.png')))
display(Image(filename=str(diagnosis / 'S3_worst_day.png')))


## Huấn luyện chính thức 300.000 bước với 5 seed

Mỗi seed khởi tạo lại từ đầu; checkpoint pilot không được dùng để cộng
thêm bước vào kết quả chính thức. Mỗi 10.000 bước có đánh giá validation
và snapshot resume; `best.pt` chọn bằng chi phí validation thấp nhất.

Sau khi Colab ngắt phiên, kết nối lại và chạy từ đầu với cùng
`EXPERIMENT_NAME`. Không đổi cấu hình hay CPU/CUDA giữa một seed.
Chỉ tiến độ chưa kịp lưu sau snapshot gần nhất phải chạy lại.


In [ ]:
# 8. Chạy đủ seed 0–4 và tổng hợp validation; không dùng test để học.
if STAGE in ('all', 'final'):
    print('Bắt đầu ô 8: kiểm tra cấu hình đã khóa, sau đó chạy/tiếp tục từng seed.', flush=True)
    print('Thư mục kết quả:', OUTPUT, '| Thiết bị:', DEVICE, flush=True)
    final_validation = workflow.run_final(OUTPUT, DEVICE)
    display(final_validation.loc[final_validation.scenario.eq('S5'),
            ['method', 'seed_count', 'adjusted_total_cost_vnd_mean',
             'adjusted_total_cost_vnd_std_seeds', 'clipped_steps_mean']])


In [ ]:
# 9. Xem reward/chi phí validation của từng seed trong suốt quá trình học.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
count = 0
for seed in range(5):
    log = OUTPUT / f'final/custom_seed{seed}/validation.csv'
    if not log.is_file():
        continue
    frame = pd.read_csv(log)
    axes[0].plot(frame.step, frame.episode_reward_mean, label=f'seed {seed}')
    axes[1].plot(frame.step, frame.adjusted_total_cost_vnd_mean, label=f'seed {seed}')
    count += 1
axes[0].set_title('Validation reward / day')
axes[1].set_title('Adjusted total cost / day')
for axis in axes:
    axis.set_xlabel('Training steps')
    if count:
        axis.legend()
plt.tight_layout()
plt.show()


## Đánh giá test sau khi cấu hình và mô hình đã được khóa

Ô dưới kiểm tra đủ 300.000 bước × 5 seed, đúng mã nguồn, dữ liệu và cấu
hình; sau đó khóa hash từng `best.pt`. Test gồm 56 ngày thuộc thời gian
khác với train/validation. S1–S5 có ngày trùng nhau; không cộng chúng
thành các ngày lịch độc lập. Không chọn lại mô hình sau khi xem test.


In [ ]:
# 10. Test chỉ chạy khi toàn bộ điều kiện của thí nghiệm đã đạt.
if STAGE in ('all', 'test'):
    final_test = workflow.evaluate_final(OUTPUT, 'test')
    display(final_test[['method', 'scenario', 'seed_count',
                        'adjusted_electric_cost_vnd_mean',
                        'adjusted_total_cost_vnd_mean',
                        'adjusted_total_cost_vnd_std_seeds',
                        'adjusted_cost_saving_pct_mean', 'clipped_steps_mean']])


In [ ]:
# 11. Đóng gói mô hình và kết quả để Dương tổng hợp; không cần gửi resume.pt lớn.
if (OUTPUT / 'final_test/summary.csv').is_file():
    import zipfile
    # Báo cáo lấy số thực đã chạy; không điền trước kết quả 300.000 bước.
    result = workflow.evaluate_final(OUTPUT, 'test')
    locked = workflow.verify_lock(OUTPUT)
    report_lines = ['# Kết quả SAC của Nguyễn Văn Trường', '',
        'Ngân sách: 300.000 bước × 5 seed. Tập báo cáo: test, sau khi khóa cấu hình.',
        'Thiết bị huấn luyện: CUDA (GPU). Chi tiết phần cứng: device_choice.json.', '',
        'Cách đặt mục tiêu đã chọn bằng validation: ' + locked['selected'] + '.', '',
        '| Phương pháp | Kịch bản | Số seed/mốc | Tổng chi phí điều chỉnh/ngày | Độ lệch chuẩn giữa seed |',
        '| --- | --- | ---: | ---: | ---: |']
    for row in result.itertuples():
        report_lines.append(f'| {row.method} | {row.scenario} | {row.seed_count} | '
            f'{row.adjusted_total_cost_vnd_mean:,.0f} | '
            f'{row.adjusted_total_cost_vnd_std_seeds:,.0f} |')
    report_lines += ['',
        'Dữ liệu và giá điện là mô phỏng. Chi phí điều chỉnh có định giá lượng điện pin dùng ròng;',
        'tổng chi phí còn có hao mòn và các khoản phạt. Không gọi đây là hóa đơn thực tế.',
        'SAC có 5 seed huấn luyện; No-battery và Rule-based chỉ cần một mốc với nhiễu đánh giá cố định.',
        'S1–S5 có ngày trùng nhau. S4 chỉ gây nhiễu quan sát, chỉ số PV chưa truy vết nguồn điện.',
        'Xem SAC_GHI_CHU_CHO_DUONG.md để biết reward, gamma, terminal và giới hạn so sánh.']
    report = OUTPUT / 'BAO_CAO_KET_QUA_SAC.md'
    report.write_text(chr(10).join(report_lines), encoding='utf-8')
    handoff = OUTPUT / 'SAC_ban_giao_Duong.zip'
    paths = [OUTPUT / name for name in ('experiment_plan.json', 'configuration_lock.json',
             'locked_config.yaml', 'device_choice.json', 'test_models_lock.json', 'pilot_comparison.csv',
             'BAO_CAO_KET_QUA_SAC.md')]
    for seed in range(5):
        folder = OUTPUT / f'final/custom_seed{seed}'
        paths += [folder / name for name in ('best.pt', 'metadata.json',
                  'training_summary.json', 'training.csv', 'validation.csv', 'learning_curve.png')]
    paths += list((OUTPUT / 'final_test').glob('*'))
    paths += list((OUTPUT / 'final_validation').glob('*'))
    temporary = handoff.with_suffix('.tmp')
    with zipfile.ZipFile(temporary, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
        archive.write(WORKSPACE / 'data/processed.csv', 'data/processed.csv')
        archive.write(WORKSPACE / 'docs/SAC_COLAB_NGUYEN_VAN_TRUONG.md',
                      'SAC_GHI_CHU_CHO_DUONG.md')
        for path in paths:
            if path.is_file():
                archive.write(path, str(path.relative_to(OUTPUT)))
    temporary.replace(handoff)
    print('Gói bàn giao đã lưu trong Drive:', handoff)
print('Log huấn luyện: final/custom_seedN/training.csv')
print('Log validation: final/custom_seedN/validation.csv')
print('Kết quả cuối: final_test/summary.csv và daily_metrics.csv')
print('Tất cả nằm trong:', OUTPUT)
